In [4]:
import torch
import torch.nn as nn
import math

In [6]:
class MultiHeadAttention(nn.Module):
  def __init__(self, model_dim, num_heads=6):
    super().__init__()
    self.WQ = nn.Linear(model_dim, model_dim) # num_heads * head_dim = model_dim
    self.WK = nn.Linear(model_dim, model_dim)
    self.WV = nn.Linear(model_dim, model_dim)
    self.WO = nn.Linear(model_dim, model_dim)
    self.model_dim = model_dim
    self.num_heads = num_heads
    self.head_dim = model_dim // num_heads

    assert model_dim % num_heads == 0

  def forward(self, q, k, v):
    B, L, _ = q.shape

    Q = self.WQ(q) # [B, Seq_len, model_dim]
    Q = torch.reshape(Q, (B, L, self.num_heads, self.head_dim)) # [B, Seq_len, num_heads, heads_dim]
    Q = torch.permute(Q, (0, 2, 1, 3)) # [B, num_heads, seq_len, heads_dim]

    K = self.WK(k)
    K = torch.reshape(K, (B, L, self.num_heads, self.head_dim))
    K = torch.permute(K, (0, 2, 1, 3))

    V = self.WV(v)
    V = torch.reshape(V, (B, L, self.num_heads, self.head_dim))
    V = torch.permute(V, (0, 2, 1, 3))

    scores = torch.matmul(Q, K.transpose(-2, -1)) # [B, num_heads, L, L]

    scores = scores / math.sqrt(self.head_dim)

    out = torch.softmax(scores, dim=-1)

    final = torch.matmul(out, V) # [B, num_heads, seq_len, heads_dim]

    final = torch.permute(final, (0, 2, 1, 3)) # [B, seq_len, num_heads, heads_dim]

    final = torch.reshape(final, (B, L, self.model_dim)) #[B, seq_len, model_dim]

    return self.WO(final)

att = MultiHeadAttention(516)